In [ ]:
import jax
import jax.random as jrandom
from orbit_wars_jax import setup, step, EnvAction
from jax_visualizer import jax_states_to_kaggle_env

# 1. Run your JAX environment and collect states
states = []
key = jrandom.PRNGKey(43)
state, params = setup(key)

# (Optional: use the random agent from adapter to generate actions, or just pass no-ops)
import jax.numpy as jnp
from orbit_wars_jax import MAX_BODIES, MAX_FLEETS_PER_PLANET_PER_STEP
noop = EnvAction(
    ships=jnp.zeros((MAX_BODIES, MAX_FLEETS_PER_PLANET_PER_STEP), dtype=jnp.int32),
    angle=jnp.zeros((MAX_BODIES, MAX_FLEETS_PER_PLANET_PER_STEP), dtype=jnp.float32)
)

# Step 100 times for example
for i in range(500):
    states.append(state)
    state, _, _ = step(state, params, noop)

# 2. Convert to Kaggle format and render!
env = jax_states_to_kaggle_env(states, params)
env.render(mode="ipython")

In [ ]:
import jax
import jax.numpy as jnp
from orbit_wars_jax import setup, step, EnvAction, MAX_BODIES, MAX_FLEETS_PER_PLANET_PER_STEP

@jax.jit
def nearest_planet_sniper_jax(state, params, player_id):
    my_planets = (state.planet_owners == player_id)

    # Target ANY active body we don't own (including neutral planets and comets)
    active_bodies = params.is_orbiting_planet | params.is_static_planet | params.is_comet
    target_planets = active_bodies & (state.planet_owners != player_id)

    # Calculate distances between all planets
    coords = state.planet_coords
    dx = coords[:, 0, None] - coords[None, :, 0]
    dy = coords[:, 1, None] - coords[None, :, 1]
    dist = jnp.sqrt(dx**2 + dy**2)

    # Mask out non-target planets by setting their distance to infinity
    dist = jnp.where(target_planets[None, :], dist, jnp.inf)

    # Find the nearest target planet for each planet
    nearest_idx = jnp.argmin(dist, axis=1)

    # How many ships do we need? Target's garrison + 1, minimum 20
    target_ships = state.planet_ships[nearest_idx]
    ships_needed = jnp.maximum(target_ships + 1, 20)

    # Only launch if we own the planet, have enough ships, and there is actually a valid target
    can_launch = my_planets & (state.planet_ships >= ships_needed) & jnp.any(target_planets)

    # Calculate the angle to the nearest target
    target_coords = coords[nearest_idx]
    my_coords = coords
    angle_to_target = jnp.arctan2(target_coords[:, 1] - my_coords[:, 1],
                                  target_coords[:, 0] - my_coords[:, 0])

    # Build the action arrays
    ships_action = jnp.zeros((MAX_BODIES, MAX_FLEETS_PER_PLANET_PER_STEP), dtype=jnp.int32)
    angle_action = jnp.zeros((MAX_BODIES, MAX_FLEETS_PER_PLANET_PER_STEP), dtype=jnp.float32)

    # We use the first fleet slot (index 0) per planet
    ships_to_launch = jnp.where(can_launch, ships_needed, 0)

    ships_action = ships_action.at[:, 0].set(ships_to_launch)
    angle_action = angle_action.at[:, 0].set(angle_to_target)

    return EnvAction(ships=ships_action, angle=angle_action)

# --- How to test it in your JAX loop ---
states = []
state, params = setup(key)

for i in range(500):
    states.append(state)

    # Player 0 and Player 1 both use the sniper agent (pass params as well!)
    actions0 = nearest_planet_sniper_jax(state, params, 0)
    actions1 = nearest_planet_sniper_jax(state, params, 1)

    # Combine the actions (since ships=0 for planets they don't own, adding them is safe)
    combined_ships = actions0.ships + actions1.ships
    combined_angles = actions0.angle + actions1.angle
    action = EnvAction(ships=combined_ships, angle=combined_angles)

    state, _, _ = step(state, params, action)

env = jax_states_to_kaggle_env(states, params)
env.render(mode="ipython")

In [2]:
from kaggle_environments import make
env = make("orbit_wars", debug=True)
env.run(["random", "random"])
env.render()

In [3]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    print()
    print(obs)
    print()
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper_bad(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets]# if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [ ]:
# Test it against the random agent
env = make("orbit_wars", debug=False)
env.run([nearest_planet_sniper, nearest_planet_sniper_bad])

final = env.steps[-1]
# for i, s in enumerate(final):
#     print(f"Player {i}: reward={s.reward}, status={s.status}")

env.render(mode="ipython", width=800, height=600)